# RSNA Knee — Kaggle runner

Thin wrapper. All logic lives in the repo's `src/`, which arrives here as an
attached Kaggle Dataset imported from GitHub.

**Order matters:** refresh the dataset on Kaggle *before* re-running, or this
executes the previous revision. Cell 2 prints the revision so a stale run is
obvious instead of silent.

Set `SCRIPT` in the last cell to pick the job:

| `SCRIPT`  | what it does                                   | Internet | Accelerator |
|-----------|------------------------------------------------|----------|-------------|
| `extract` | report → 12 labels, one time over 4407 reports  | **on**   | GPU T4 x2   |
| `eval`    | grade those labels against the 58 gold studies  | off      | CPU is fine |
| `train`   | image model → `submission.csv`                  | off      | GPU T4 x2   |

`extract` must run *before* `eval`, and `eval` reads the CSV from
`/kaggle/working`, which does not survive the session — so run them together,
or save the output (see the last cell).

Use T4, not P100 — Kaggle's PyTorch ships without Pascal kernels.


In [ ]:
# Find the code. Normally it arrives as an attached Kaggle Dataset, so search
# /kaggle/input first. Kaggle nests inputs by type
# (/kaggle/input/competitions/..., /kaggle/input/datasets/<owner>/...), so this
# walks a few levels instead of guessing a fixed path. The walk is depth-capped
# and prunes the DICOM trees on purpose: an unbounded walk would crawl tens of
# thousands of series directories before giving up.
#
# If nothing is attached, clone from GitHub instead. That works while Internet
# is on -- which the extract job needs anyway -- and means a code change reaches
# Kaggle by a plain `git push`. It will NOT work for the offline submission run,
# which still needs the dataset attached.
import os
import subprocess

REPO_URL = "https://github.com/xuanxiaoyuan/RSNA-Knee-Abnormality-Detection.git"
MARKER = "src/train.py"
HEAVY = {"train_series", "test_series", "train", "test"}


def find_repo(root, max_depth=4):
    base_depth = root.rstrip("/").count("/")
    for dirpath, dirnames, _ in os.walk(root):
        if dirpath.rstrip("/").count("/") - base_depth >= max_depth:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if d not in HEAVY]
        if os.path.exists(os.path.join(dirpath, MARKER)):
            return dirpath
    return None


print("attached under /kaggle/input:")
for entry in sorted(os.listdir("/kaggle/input")):
    path = f"/kaggle/input/{entry}"
    if os.path.isdir(path):
        print(f"  {entry}/  ({len(os.listdir(path))} entries)")

CODE = find_repo("/kaggle/input")

if CODE is None:
    CODE = "/kaggle/working/rsne-code"
    if os.path.exists(f"{CODE}/.git"):
        print(f"\n[code] refreshing clone at {CODE}")
        subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=True)
    else:
        print(f"\n[code] nothing attached; cloning {REPO_URL}")
        try:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
        except subprocess.CalledProcessError:
            raise SystemExit(
                "[code] git clone failed. Either enable Internet "
                "(Settings -> Internet) or attach the code dataset via Add Input."
            )

print("\nCODE =", CODE)


In [ ]:
# Staleness check: does this match the VERSION you just pushed?
print(open(f"{CODE}/VERSION").read())

In [ ]:
# Pick the job and run it. Each entry point reads its own config from the repo.
SCRIPT = "train"   # "extract" | "eval" | "train"
LIMIT = None       # extract only: int to label just the first N reports

import glob, os, runpy, shutil, sys

# Re-running a cell does NOT reload imported modules: Python caches them in
# sys.modules and cell 1's git pull only changes files on disk. Without this a
# second run in the same kernel mixes revisions — train.py is re-executed by
# runpy and picks up the new code while dataset/model/utils stay at whatever
# was imported first, which surfaces as an error deep inside the run.
for name in ("dataset", "model", "utils", "extract_labels", "eval_extractor"):
    stale = sys.modules.pop(name, None)
    if stale is not None:
        print(f"[cache] dropped stale {name}: {getattr(stale, '__file__', '?')}")

# train.py reads the derived labels from work_dir, but the dataset they arrive
# in is read-only, so copy the file across first. Kaggle nests inputs by type,
# hence the two globs.
labels_src = (glob.glob("/kaggle/input/datasets/*/*/labels_derived.csv")
              or glob.glob("/kaggle/input/*/labels_derived.csv"))
if SCRIPT == "train":
    if labels_src:
        os.makedirs("/kaggle/working", exist_ok=True)
        shutil.copy(labels_src[0], "/kaggle/working/labels_derived.csv")
        print(f"[input] labels_derived.csv <- {labels_src[0]} "
              f"({os.path.getsize(labels_src[0])} bytes)")
    else:
        print("[input] no labels_derived.csv attached; training on gold only")

if SCRIPT == "extract":
    # Needs Internet ON for the model weights. Raise LIMIT in steps (20, 300,
    # then None) so a bad prompt shows up in minutes, not after the full run.
    argv = ["extract_labels.py", "--config", f"{CODE}/configs/extract.yaml"]
    if LIMIT is not None:
        argv += ["--limit", str(LIMIT)]
elif SCRIPT == "eval":
    argv = ["eval_extractor.py", "--config", f"{CODE}/configs/extract.yaml"]
elif SCRIPT == "train":
    argv = ["train.py", "--config", f"{CODE}/configs/base.yaml", "--no-debug"]
else:
    raise SystemExit(f"unknown SCRIPT {SCRIPT!r} - use 'extract', 'eval' or 'train'")

print("running", *argv)
sys.argv = argv
runpy.run_path(f"{CODE}/src/{argv[0]}", run_name="__main__")


## `extract` — the one-time label job

1. Internet **on**, accelerator T4 x2.
2. Run a smoke test (`LIMIT = 20`, then 300) and read the printed
   `[parse] named=… partial=… fail=…` line and the s/report timing before
   committing to the full run. `fail` counts reports where *nothing* parsed;
   `partial` counts reports where only some findings parsed. Either one above
   zero means the prompt needs a look before the full run.
3. Set `LIMIT = None` for the full 4407 and run it as a **commit**
   (Save Version → Save & Run All), not interactively: `/kaggle/working` does
   not survive a session, and an interactive session dies when the browser
   disconnects. `resume: true` skips studies already in the output file, so a
   killed run continues rather than restarts — but only if the file survived,
   which is what committing guarantees.
4. The finished CSV must end up as a Kaggle Dataset, because the training
   notebook reads it as input.

## `train` — after the debug run is clean

1. Attach that Dataset, then run this cell: it copies `labels_derived.csv`
   into the work dir for you.
2. Confirm the `[cache]` / `[input]` / `[labels]` lines — `training on N/4407`
   with N near 4407 means the derived labels reached training.
3. Run as a **commit** too. It is a multi-hour job; the checkpoint is written
   after every epoch, so a killed run still leaves the weights behind, and each
   epoch prints its own `val_macro_auc`.
